# Logistic Regression

In [2]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report


df = pd.read_csv("pca_features.csv")

TARGET = "chronic_absent"

## 1. Feature Groups

In [3]:
BASE_CAT = [
    "Gen", "Eth", "Fluency", "SpEd",
    "SiteName", "Zip", "Grade", "school_stage",
    "SED"
]
BASE_NUM = [
    "age", "CurrWeightedTotGPA", "Susp", "DaysEnr",
    "year",
]

LEAKAGE = ["DaysAbs", "AttRate", "absence_rate", "gpa_absence"]

# socio / crime（raw features）

RAW_CRIME = [
    "total_crimes", "violent_crimes", "property_crimes", "drug_crimes", "other_crimes",
    "crime_rate", "violent_crime_rate", "property_crime_rate", "drug_crime_rate", "other_crime_rate",
]
RAW_SOCIO = [
    "total_population", "poverty_rate_pct", "median_household_income", "unemployment_rate_pct",
    "high_school_plus_rate_pct", "college_degree_rate_pct",
    "median_gross_rent", "median_home_value", "uninsured_rate_pct",
    "poverty_crime",
]

BASE_NUM_CLEAN = [c for c in BASE_NUM if c not in LEAKAGE]
BASE_CAT_CLEAN = [c for c in BASE_CAT if c not in LEAKAGE]
PCA_IDX = ["socio_index", "crime_index"]

GROUPS = {
    "raw_only": {
        "num": BASE_NUM_CLEAN + RAW_SOCIO + RAW_CRIME,
        "cat": BASE_CAT_CLEAN,
    },
    "pca_only": {
        "num": BASE_NUM_CLEAN + PCA_IDX,
        "cat": BASE_CAT_CLEAN,
    },
    "both": {
        "num": BASE_NUM_CLEAN + RAW_SOCIO + RAW_CRIME + PCA_IDX,
        "cat": BASE_CAT_CLEAN,
    },
}

## 2. Temporal Split

In [4]:
df = df.dropna(subset=[TARGET]).copy()
cutoff = 2122
train_df = df[df["year"] <= cutoff].copy()
test_df  = df[df["year"] > cutoff].copy()

## 3. Experiments

In [8]:
import experiments 
GROUPS = {
    "raw_only": RAW_SOCIO + RAW_CRIME,
    "pca_only": PCA_IDX,
    "both": RAW_SOCIO + RAW_CRIME + PCA_IDX,
}

rows = []
for name, community_cols in GROUPS.items():
    m = experiments.run_logit_experiment(
        train=train_df,
        test=test_df,
        target=TARGET,
        base_num=BASE_NUM,
        base_cat=BASE_CAT,
        community_cols=community_cols,
        leakage=[],
    )
    rows.append({"group": name, **m})

results = pd.DataFrame(rows)
results[["group","auc_roc","auc_pr","precision","recall","f1","recall_at_top_k"]]

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_ite

,group,auc_roc,auc_pr,precision,recall,f1,recall_at_top_k
0,raw_only,0.545760,0.458836,0.460134,0.852099,0.597577,0.105418
1,pca_only,0.649150,0.595105,0.440430,0.969752,0.605749,0.169681
2,both,0.546646,0.461765,0.460379,0.858235,0.599286,0.110549
